In [2]:
import sys
print(sys.executable)

D:\python\python.exe


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.7.1+cu118
True


In [4]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [5]:
from torchvision.datasets import MNIST##本地数据库
from torchvision import transforms##之后学一下transforms

dataset = MNIST(
    root="data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

In [6]:
from torch.utils.data import DataLoader##DataLoader = 自动搬运工，把 Dataset 包装成“自动分批的数据流”

bs = 64#一次训练样本量
loader = DataLoader(dataset, batch_size=bs, shuffle=True)##shuffle洗牌，打乱顺序，自动分 batch=64，
##自动迭代不用for in循环

xb, yb = next(iter(loader))##创建一个“数据读取器”，光标指向第一批数据，next（）取下一批，
##等价于for xb, yb in loader：，xb是图片，yb是label
print(xb.shape, yb.shape)
print(xb.dtype, yb.dtype)


torch.Size([64, 1, 28, 28]) torch.Size([64])
torch.float32 torch.int64


In [7]:
xb = xb.view(xb.size(0), -1)##展开为一维向量
print(xb.shape, yb.shape)

torch.Size([64, 784]) torch.Size([64])


In [12]:
from torch import nn
import torch.nn.functional as F
class MNIST_NN(nn.Module):##网络结构,前边权重可以手动随机化，这里用torch的功能自动配置好了，
    ##nn.Module输入是一张图片，pred是经过模型的输出
    def __init__(self):
        super().__init__()
        self.hidden1=nn.Linear(784,128)##784个变量转化为128个特征进入第二层
        self.hidden2=nn.Linear(128,256)
        self.out=nn.Linear(256,10)
        self.dropout=nn.Dropout(0.5)##随机杀死节点防止过度拟合
    def forward(self,x):##x在此处是64*784，此功能用于前向传播，relu是激活
        x=F.relu(self.hidden1(x))
        x=F.relu(self.dropout(x))
        x=F.relu(self.hidden2(x))##先linear再relu
        x=F.relu(self.dropout(x))##只有训练时会dropout，测试不会
        x=self.out(x)
        return x

In [13]:
net=MNIST_NN()

In [14]:
print(net)

MNIST_NN(
  (hidden1): Linear(in_features=784, out_features=128, bias=True)
  (hidden2): Linear(in_features=128, out_features=256, bias=True)
  (out): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [15]:
pred = net(xb)
print(pred.shape, yb.shape, yb.dtype)

torch.Size([64, 10]) torch.Size([64]) torch.int64


In [16]:
loss = F.cross_entropy(pred, yb)
print(loss.item())

2.3361997604370117


In [12]:
net.zero_grad()
loss.backward()

print(net.hidden1.weight.grad is None)   # 应该是 False
print(net.hidden1.weight.grad.abs().mean().item())  # 应该是一个 >0 的小数


False
0.000596012978348881


In [13]:
import torch.optim as optim
opt = optim.SGD(net.parameters(), lr=0.1)


In [14]:
# 更新前loss
pred = net(xb)
loss0 = F.cross_entropy(pred, yb)

opt.zero_grad()
loss0.backward()
opt.step()

# 更新后loss（同一批数据）
pred = net(xb)
loss1 = F.cross_entropy(pred, yb)

print(loss0.item(), loss1.item())


2.312959671020508 2.305136203765869


In [23]:
from torch import optim

net = MNIST_NN()##初始化，目前是随机的参数

optimizer = optim.SGD(net.parameters(), lr=0.1)##优化器，lr是学习率，根据梯度更新模型参数

for epoch in range(3):##完整看完整个数据集一次*3

    for xb, yb in loader:##每次拿一批(batch)数据，然后pytorch自带的loader会往后推

        xb = xb.view(xb.size(0), -1)##展平，可以先将数据集展平再训练会变快

        pred = net(xb)##前向传播，实际等同于pred = net.forward(xb)，preb现在结构[64,10]，64张图片，每个有0-9的评分
        ##调用 Module.__call__()，内部自动执行 forward，并构建计算图

        loss = F.cross_entropy(pred, yb)##计算交叉熵损失

        loss.backward()##反向传播，计算梯度

        optimizer.step()##参数 = 参数 - 学习率 × 梯度

        optimizer.zero_grad()##清0，PyTorch 梯度默认累加，必须手动清零

    print("epoch done")

epoch done
epoch done
epoch done


In [21]:
nettwo = MNIST_NN()##初始化

optimizertwo = optim.SGD(net.parameters(), lr=0.1)##优化器，lr是学习率，优化器把参数加载进去，然后计算返回给模型

print(optimizertwo)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [24]:
xb, yb = next(iter(loader))
xb = xb.view(xb.size(0), -1)

pred = net(xb)
print(pred[0])

tensor([-4.5059,  2.4288,  2.5692, -0.5508, -1.4858,  1.5382,  7.7731, -4.9455,
         0.0872, -2.4768], grad_fn=<SelectBackward0>)


In [27]:
yb##这里第0个数是6，pred0中的6对应7.7731，是最大的，成功

tensor([6, 0, 7, 5, 1, 0, 3, 1, 6, 7, 9, 8, 4, 2, 9, 3, 6, 7, 6, 7, 4, 8, 8, 3,
        4, 1, 1, 8, 0, 5, 2, 7, 6, 2, 1, 6, 2, 9, 8, 4, 5, 9, 7, 0, 7, 1, 4, 9,
        9, 8, 8, 4, 0, 3, 3, 9, 1, 4, 5, 5, 9, 7, 0, 1])

In [28]:
print(pred[3])

tensor([-2.7628,  0.2904, -3.0072,  0.8304,  0.3090,  6.6345, -0.2070, -3.1501,
         1.9334, -1.0075], grad_fn=<SelectBackward0>)


In [29]:
##由于有展平的一步，全连接（MLP）不能理解二维结构，纯属数学，所以需要卷积（CNN）